In [ ]:

from analysis.types import PositionsDict
from analysis.vacf import compute_velocity
from numpy.typing import NDArray
from analysis.trajectories import load_positions
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
trajectories,*_ = load_positions('./output_droplet_tracking/csv_files/for_2.3_droplets.csv')


In [ ]:


def compute_vacf(
    r: NDArray[np.floating],
    lagtime: int = None,
    dt: float = 1.0,
) -> NDArray[np.floating]:
    """
    Compute the velocity autocorrelation function for a single trajectory.
    Normalized VACF.
    Parameters
    ----------
    r : ndarray, shape (N, 2)
        Ordered (x, y) positions.
    dt : float
        Time step between consecutive frames.

    Returns
    -------
    vacf : ndarray, shape (N-1,)
        VACF values at lags τ = 0, 1, …, N-2 (in frame units).
        ``vacf[0] = <v·v>`` (mean kinetic energy per unit mass × 2).

    Notes
    -----
    Velocity is computed with ``compute_velocity`` (forward difference).
    The dot-product estimator is used:
    ``VACF(τ) = mean_t [vx(t)·vx(t+τ) + vy(t)·vy(t+τ)]``.
    """
    v = compute_velocity(r, dt)
    if lagtime is not None:
        v = v[:lagtime]
        n = lagtime
    else:
        n = len(v)
    
    vacf = np.zeros(n)
    for tau in range(n):
        corr = v[:n - tau, 0] * v[tau:, 0] + v[:n - tau, 1] * v[tau:, 1]
        vacf[tau] = np.mean(corr)
    return vacf



def compute_cross_vcaf(v1:NDArray[np.floating],
                    v2:NDArray[np.floating],
                    lag_time:int = None)->tuple[NDArray[np.floating],int]:

    vi = compute_velocity(v1, dt=0.2)
    vj = compute_velocity(v2, dt=0.2)

    n = min(len(vi), len(vj))

    if lag_time is None:
        lag_time = n - 1
        
    cij = np.zeros(lag_time)

    norm_i = np.mean(vi[:,0]**2 + vi[:,1]**2)
    norm_j = np.mean(vj[:,0]**2 + vj[:,1]**2)

    norm = np.sqrt(norm_i * norm_j)

    for tau in range(lag_time):

        corr = (
            vi[:n-tau,0] * vj[tau:,0]
            +
            vi[:n-tau,1] * vj[tau:,1]
        )

        cij[tau] = np.mean(corr) / norm
        
    return cij,lag_time



def compute_cross_correlation_matrix_fr_all_time(trajectories: PositionsDict,lagtime:int)->NDArray[np.floating]:
    """computes a cross-correlation matrix for all tau from 0 to lagtime

    Args:
        trajectories (PositionsDict): _description_
        lagtime (int): _description_
        is_max (bool, optional): _description_. Defaults to True.

    Returns:
        NDArray[np.floating]: _description_
    """
    nsize = len(trajectories.keys())
    m = np.zeros((lagtime,nsize,nsize))

    for i,key1 in enumerate(trajectories.keys()):
        for j,key2 in enumerate(trajectories.keys()):

            vi = trajectories[key1]
            vj = trajectories[key2]
            cij,_ = compute_cross_vcaf(v1 = vi,v2= vj,lag_time=lagtime)
    
            m[:,i,j] = cij.squeeze()
       
    return m
lag_time = int(trajectories[list(trajectories.keys())[0]].shape[0]*0.4)
m = compute_cross_correlation_matrix(trajectories,lagtime=lag_time)

In [ ]:
# here index is our lagtime 
m[1]

In [ ]:
from seaborn import heatmap

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(15,5))
ax = ax.flatten()
for i in range(3):
    heatmap(m[i],xticklabels=list(trajectories.keys()),yticklabels=list(trajectories.keys()),annot=True,fmt=".2f",cmap="coolwarm",ax=ax[i])

In [ ]:
import os 
from analysis.plotting import plot_vacf_heatmap_btw_droplets


plot_vacf_heatmap_btw_droplets(corr_matrix=m,droplet_ids=list(trajectories.keys()),lag_time=0,save_plot=True,output_dir="./output_droplet_tracking/plots",label="cross_correlation")

In [ ]:
largest_singular_value1 = np.zeros(m.shape[0])

for i in range(m.shape[0]):
    U,S,VT = np.linalg.svd(m[i])
    largest_singular_value1[i] = S[0]
plt.plot(range(m.shape[0]), largest_singular_value1)

In [ ]:
largest_singular_value = np.zeros(m.shape[0])

for i in range(m.shape[0]):
    U,S,VT = np.linalg.svd(m[i])
    largest_singular_value[i] = S.max()
plt.plot(range(m.shape[0]), largest_singular_value)

In [ ]:
np.sum(largest_singular_value == largest_singular_value1)

In [ ]:
largest_singular_value = np.zeros(m.shape[0])
for i in range(m.shape[0]):
    np.fill_diagonal(m[i],0)
    U,S,VT = np.linalg.svd(m[i])
    largest_singular_value[i] = S.max()
plt.plot(range(m.shape[0]), largest_singular_value)

In [ ]:
svals = np.zeros((m.shape[0], 6))

for t in range(m.shape[0]):
    _,S,_ = np.linalg.svd(m[t])
    svals[t] = S

for k in range(6):
    # plt.plot(svals[:,k],label=f'Singular Value {k+1}')
    psd = np.abs(np.fft.rfft(svals[:,k]))**2
    psd_norm = psd / psd.max()
    plt.plot(psd_norm,label=f'Singular Value {k+1}')
    plt.ylabel("Normalized PSD")
    plt.ylim(-.1,.15)
    plt.xlim(-1,20)
    
    plt.legend()

In [ ]:
for k in range(6):

    x = svals[:,k]

    x = x - np.mean(x)

    psd = np.abs(np.fft.rfft(x))**2

    psd /= psd.max()

    plt.plot(psd,label=f'S{k+1}')

In [ ]:
for k in range(6):

    x = svals[:,k]
    x -= np.mean(x)

    psd = np.abs(np.fft.rfft(x))**2

    freq = np.fft.rfftfreq(len(x), d=0.2)

    plt.semilogy(freq[1:], psd[1:], label=f'S{k+1}')

plt.legend()

In [ ]:
sigmas = []

for tau in range(m.shape[0]):
    A = m[tau].T @ m[tau]
    eigvals, _ = np.linalg.eigh(A)
    eigvals = eigvals[::-1]
    sigmas.append(np.sqrt(np.maximum(eigvals, 0)))

sigmas = np.array(sigmas)

In [ ]:
for i in range(sigmas.shape[1]):
    plt.plot(sigmas[:, i], label=f'Sigma {i+1}')
    plt.legend()

In [ ]:
energy = sigmas**2
energy /= energy.sum(axis=1, keepdims=True)

for i in range(energy.shape[1]):
    plt.plot(energy[:, i], label=f"Mode {i+1}")

plt.xlabel("Lag")
plt.ylabel("Fraction of total correlation")
plt.legend()

In [ ]:

energy = sigmas**2
energy /= energy.sum(axis=1, keepdims=True)

r_eff = 1 / np.sum(energy**2, axis=1)

plt.figure(figsize=(7,4))
plt.plot(r_eff)
plt.xlabel("Lag")
plt.ylabel("Effective rank")
plt.grid(True)
tau = 1200

U, S, VT = np.linalg.svd(m[tau], full_matrices=False)

print(S)
print(VT[0])   # dominant right singular vector
print(VT[1])   # second mode

In [ ]:
tau = 1200
ids = list(trajectories.keys())
U, S, VT = np.linalg.svd(m[tau], full_matrices=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].bar(ids, VT[0])
ax[0].set_title("Mode 1")
ax[0].set_ylabel("Weight")

ax[1].bar(ids, VT[1])
ax[1].set_title("Mode 2")

plt.tight_layout()

In [ ]:
overlap = []

prev = None

for tau in range(m.shape[0]):
    _, _, VT = np.linalg.svd(m[tau], full_matrices=False)
    v = VT[0]

    if prev is not None:
        overlap.append(abs(prev @ v))

    prev = v

plt.figure(figsize=(7, 4))
plt.plot(overlap)
plt.xlabel("Lag")
plt.ylabel(r"$|v_1(\tau)^T v_1(\tau+1)|$")
plt.grid(True)

In [ ]:
gap = sigmas[:, 0] - sigmas[:, 1]

plt.figure(figsize=(7, 4))
plt.plot(gap)
plt.xlabel("Lag")
plt.ylabel(r"$\sigma_1 - \sigma_2$")
plt.grid(True)

In [ ]:
overlap = []

prev = None

for tau in range(m.shape[0]):
    _, _, VT = np.linalg.svd(m[tau], full_matrices=False)

    # dominant 2D subspace
    V = VT[:2].T          # shape (N, 2)

    P = V @ V.T           # projection matrix

    if prev is not None:
        overlap.append(np.trace(prev @ P) / 2)

    prev = P

plt.figure(figsize=(7,4))
plt.plot(overlap)
plt.xlabel("Lag")
plt.ylabel("2D subspace overlap")
plt.grid(True)

In [ ]:
taus = [0, 600, 1200, 1500]

fig, ax = plt.subplots(1, len(taus), figsize=(16, 4))

for k, tau in enumerate(taus):
    _, _, VT = np.linalg.svd(m[tau], full_matrices=False)
    V = VT[:2].T
    P = V @ V.T

    im = ax[k].imshow(P, cmap="coolwarm", vmin=-1, vmax=1)
    ax[k].set_title(f"Lag = {tau}")
    ax[k].set_xticks(range(len(ids)))
    ax[k].set_xticklabels(ids)
    ax[k].set_yticks(range(len(ids)))
    ax[k].set_yticklabels(ids)

plt.colorbar(im, ax=ax)

In [ ]:
tau = 1200

U,S,VT = np.linalg.svd(m[tau], full_matrices=False)

C1 = S[0] * np.outer(U[:,0], VT[0])

fig,ax = plt.subplots(1,2, figsize=(8,4))

heatmap(m[tau], ax=ax[0], cmap="coolwarm", vmin=-1, vmax=1)
ax[0].set_title("Original")

heatmap(C1, ax=ax[1], cmap="coolwarm", vmin=-1, vmax=1)
ax[1].set_title("Rank-1 approximation")